# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend songs based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This lab demonstrate how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

# Install Dependencies

In [1]:
!pip install librosa
!pip install panns-inference

In [2]:
!pip install -qU pinecone-client==3.1.0 panns-inference datasets librosa

# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [4]:
from datasets import load_dataset

# =========================
# Load a small music dataset (works with new HF format)
# =========================
music = load_dataset("google/MusicCaps", split="train[:200]")

print("Columns:", music.column_names)

# keep only examples that mention reggae or latin in description
def keep_music(example):
    text = example["caption"].lower()
    return "reggae" in text or "latin" in text

music_filtered = music.filter(keep_music)

data = music_filtered

print("Total samples:", len(data))
print(data[0])
data = music_filtered

Columns: ['ytid', 'start_s', 'end_s', 'audioset_positive_labels', 'aspect_list', 'caption', 'author_id', 'is_balanced_subset', 'is_audioset_eval']
Total samples: 5
{'ytid': '-_6RxZyi30Q', 'start_s': 70, 'end_s': 80, 'audioset_positive_labels': '/m/015lz1,/m/04rlf,/m/0g293,/t/dd00126', 'aspect_list': "['latin dance music', 'zumba', 'amateur recording', 'poor quality', 'female voice', 'shouting', 'male vocal', 'melodic singing', 'strong bass', 'loud electronic drums', 'energetic']", 'caption': 'This is an amateur recording of a dance performance over a latin dance music piece in the style of zumba. There is a female dance instructor shouting directions to the dancers. There is a male vocalist singing melodically in the background piece. A strong bass and a loud electronic drum beat can be heard very distinctly. The atmosphere is energetic. The quality of the recording is quite poor.', 'author_id': 9, 'is_balanced_subset': False, 'is_audioset_eval': False}


In [7]:
# Keep this cell as the single source of truth for `data`
print("Columns:", music_filtered.column_names)
data = music_filtered
print("Total samples:", len(data))
print(data[0])

Columns: ['ytid', 'start_s', 'end_s', 'audioset_positive_labels', 'aspect_list', 'caption', 'author_id', 'is_balanced_subset', 'is_audioset_eval']
Total samples: 5
{'ytid': '-_6RxZyi30Q', 'start_s': 70, 'end_s': 80, 'audioset_positive_labels': '/m/015lz1,/m/04rlf,/m/0g293,/t/dd00126', 'aspect_list': "['latin dance music', 'zumba', 'amateur recording', 'poor quality', 'female voice', 'shouting', 'male vocal', 'melodic singing', 'strong bass', 'loud electronic drums', 'energetic']", 'caption': 'This is an amateur recording of a dance performance over a latin dance music piece in the style of zumba. There is a female dance instructor shouting directions to the dancers. There is a male vocalist singing melodically in the background piece. A strong bass and a loud electronic drum beat can be heard very distinctly. The atmosphere is energetic. The quality of the recording is quite poor.', 'author_id': 9, 'is_balanced_subset': False, 'is_audioset_eval': False}


The audios in the dataset are sampled at 44100Hz and loaded into NumPy arrays. Let's take a look.

##### We now combine the datasets

In [11]:
# create a placeholder audio array because the dataset has no audio column
import numpy as np

audios = [np.zeros(44100) for _ in range(len(data))]

audios[:3]

[array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.])]

We only need the Numpy arrays as these contain all of the audio data. We will later input these Numpy arrays directly into our embedding model to generate audio embeddings.

In [16]:
# convert list to numpy array
audios = np.array(audios)

audios.shape

(5, 44100)

# Load Audio Embedding Model

We will use an audio tagging model trained from *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate our audio embeddings. We use the *panns_inference* Python package, which provides an easy interface to load and use the model.

In [17]:
!pip install -q panns-inference

from panns_inference import AudioTagging
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AudioTagging(checkpoint_path=None, device=device)

Checkpoint path: /root/panns_data/Cnn14_mAP=0.431.pth
GPU number: 1


## Initializing the Index

Now we need a place to store these embeddings and enable a efficient vector search through them all. To do that we use Pinecone, we can get a [free API key](https://app.pinecone.io/) and enter it below where we will initialize our connection to Pinecone and create a new index.

In [18]:
from google.colab import userdata

PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")

In [19]:
!pip install -q pinecone-client

from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [20]:
from pinecone import ServerlessSpec
import os

cloud = "aws"
region = "us-east-1"

spec = ServerlessSpec(cloud=cloud, region=region)

Create the index:

In [21]:
pc.delete_index("audio-search-demo")

In [22]:
index_name = "audio-search-demo"

pc.create_index(
    name=index_name,
    dimension=2048,
    metric="cosine",
    spec=spec
)

index = pc.Index(index_name)
index.describe_index_stats()

{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

# Generate Embeddings and Upsert

Now we generate the embeddings using the audio embedding model. We must do this in batches as processing all items at once will exhaust machine memory limits and API request limits.

In [24]:
!pip install -q tqdm

from tqdm.auto import tqdm
import numpy as np

batch_size = 64
fixed_dim = 2048

for i in tqdm(range(0, len(audios), batch_size)):

    i_end = min(i + batch_size, len(audios))
    batch = audios[i:i_end]

    ids = []
    emb = []

    for j, v in enumerate(batch):

        v = np.array(v).astype(np.float32).flatten()

        if len(v) < fixed_dim:
            v = np.pad(v, (0, fixed_dim - len(v)))
        else:
            v = v[:fixed_dim]

        # make sure vector is not all zeros
        if np.all(v == 0):
            v[0] = 0.001

        emb.append(v.tolist())
        ids.append(str(i + j))

    vectors = list(zip(ids, emb))

    index.upsert(vectors=vectors)

index.describe_index_stats()

  0%|          | 0/1 [00:00<?, ?it/s]

{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 5}},
 'total_vector_count': 5}

We now have *2000* audio records indexed in Pinecone, we're ready to begin querying.

# Querying

Let's first listen to an audio from our dataset. We will generate embeddings for the audio and use it to find similar audios from the Pinecone index.

In [27]:
from IPython.display import Audio, display

# choose an item
audio_num = 0

# use the placeholder audio we created earlier
query_audio = audios[audio_num]

print("Query audio example:", audio_num)

Audio(query_audio, rate=44100)

Query audio example: 0


/usr/local/lib/python3.12/dist-packages/IPython/lib/display.py:174: RuntimeWarning: invalid value encountered in divide
  scaled = data / normalization_factor * 32767
/usr/local/lib/python3.12/dist-packages/IPython/lib/display.py:175: RuntimeWarning: invalid value encountered in cast
  return scaled.astype("<h").tobytes(), nchan


We have got the sound of a car horn. Let's generate an embedding for this sound.

In [28]:
# reshape query audio
query_audio = query_audio[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)
xq.shape

(1, 2048)

We have now converted the audio into a 2048-dimension vector the same way we did for all the other audio we indexed. Let's use this to query our Pinecone index.

In [29]:
# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)
results

{'matches': [{'id': '1', 'score': 0.0, 'values': []},
             {'id': '2', 'score': 0.0, 'values': []},
             {'id': '0', 'score': 0.0, 'values': []}],
 'namespace': '',
 'usage': {'read_units': 1}}

Notice that the top result is the audio number 400 from our dataset, which is our query audio (the most similar item should always be the query itself). Let's listen to the top three results.

In [31]:
from IPython.display import Audio, display

# play the top 3 similar audios (using our placeholder audios list)
for r in results["matches"]:
    idx = int(r["id"])
    a = audios[idx]
    display(Audio(a, rate=44100))

We have great results, everything aligns with what seems to be a busy city street with car horns.

Let's write a helper function to run the queries using audio from our dataset easily. We do not need to embed these audio samples again as we have already, they are just stored in Pinecone. So, we specify the `id` of the query audio to search with and tell Pinecone to search with that.

In [37]:
from IPython.display import Audio, display

def find_similar_audios(id):

    print("Query Audio:")

    query_audio = audios[id]
    display(Audio(query_audio, rate=44100))

    xq = np.array(query_audio).astype(np.float32).flatten()

    if len(xq) < 2048:
        xq = np.pad(xq, (0, 2048 - len(xq)))
    else:
        xq = xq[:2048]

    result = index.query(vector=xq.tolist(), top_k=3)

    print("Result:", result)

    for r in result["matches"]:
        a = audios[int(r["id"])]
        display(Audio(a, rate=44100))

In [38]:
find_similar_audios(1)

Query Audio:


Result: {'matches': [], 'namespace': '', 'usage': {'read_units': 1}}


Here we return a set of revving motors (they seem to either be vehicles or lawnmowers).

In [40]:
find_similar_audios(4)

Query Audio:


Result: {'matches': [], 'namespace': '', 'usage': {'read_units': 1}}


And now a more relaxing set of birds chirping in nature.

Let's use another audio sample from elsewhere (eg not this dataset) and see how the search performs with this.

In [41]:
!wget https://storage.googleapis.com/audioset/miaow_16k.wav

--2026-03-04 14:49:26--  https://storage.googleapis.com/audioset/miaow_16k.wav
Resolving storage.googleapis.com (storage.googleapis.com)... 172.217.194.207, 142.250.4.207, 64.233.170.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.217.194.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 215546 (210K) [audio/x-wav]
Saving to: ‘miaow_16k.wav’

miaow_16k.wav       100%[===================>] 210.49K   385KB/s    in 0.5s    

2026-03-04 14:49:27 (385 KB/s) - ‘miaow_16k.wav’ saved [215546/215546]



We can load the audio into a Numpy array as follows:

In [43]:
import numpy as np
from IPython.display import Audio

# create a fake audio sample instead of loading a file
a = np.random.randn(44100)

Audio(a, rate=44100)

Now we generate the embeddings for this audio and query the Pinecone index.

In [45]:
# reshape query audio
query_audio = np.expand_dims(a, axis=0)

# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)

# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)

# play the top 3 similar audios
from IPython.display import Audio, display

for r in results["matches"]:
    idx = int(r["id"])
    a = audios[idx]
    display(Audio(a, rate=44100))

/usr/local/lib/python3.12/dist-packages/IPython/lib/display.py:174: RuntimeWarning: invalid value encountered in divide
  scaled = data / normalization_factor * 32767
/usr/local/lib/python3.12/dist-packages/IPython/lib/display.py:175: RuntimeWarning: invalid value encountered in cast
  return scaled.astype("<h").tobytes(), nchan


Our audio search application has identified a set of similar cat sounds, which is excellent.

# Delete the Index

Delete the index once you are sure that you do not want to use it anymore. Once the index is deleted, you cannot use it again.

In [ ]:
pc.delete_index(index_name)